# Universidad del Gran Rosario - UGR

## Ingeniería del Software

### Trabajo Práctico Integrador

**Introducción a las metodologías y procesos de Ciencia de Datos**

**Docentes:** Ing. Ignacio Sanseovich - Lic. Briant Gauna

**Estudiantes:**

- Ivan Porcari
- Marcelo Saucedo
- Rodrigo Zamora
- Gaspar Giannitrapani
- Stefania Martos
- Juan Ignacio Sasia
- Juan José Muñoz Franchi
- Jorge Nicolas Segovia


# Estudio de caso

Este trabajo se basa en la aplicación de una metodología de minería de datos y su implementación en una solución de software.

El caso elegido se centra en el análisis de la demanda energética, tomando como punto de partida datos históricos de consumo. La idea principal es estudiar el comportamiento de la demanda a lo largo del tiempo, detectar patrones relevantes y evaluar si es posible construir un modelo que ayude a estimar períodos de mayor consumo.

Para ordenar el desarrollo del trabajo se utilizará la metodología CRISP-DM, ya que permite recorrer el proyecto desde la comprensión del problema hasta la evaluación de resultados y una posible implantación conceptual de la solución.


##Fases del Trabajo:


# Índice general

0. Configuración inicial
1. Comprensión del negocio
2. Comprensión de los datos
3. Preparación de los datos
4. Exploración de los datos
5. Modelado
6. Evaluación
7. Implantación conceptual
8. Conclusiones finales
9. Referencias y anexos


# 0. Configuración inicial




En esta sección se dejará preparado el entorno de trabajo del notebook.

Acá se van a importar las librerías necesarias, conectar Google Drive y definir las rutas principales del proyecto para trabajar siempre sobre la misma estructura de carpetas.

La idea es mantener separados los datos originales, los datos procesados y los resultados generados durante el análisis.

## 0.1 Importación de librerías

En esta parte importamos las librerías básicas que vamos a usar para trabajar con archivos, rutas y datos tabulares.


In [1]:
from google.colab import files, userdata
import pandas as pd
from pathlib import Path

## 0.2 Verificación de acceso a la carpeta del proyecto

Como el proyecto está en una carpeta compartida de Google Drive, primero verificamos que Colab pueda acceder correctamente a la carpeta raíz del proyecto.

_Para que esta ruta funcione, la carpeta compartida **TPI** debe estar agregada como acceso directo dentro de **Mi unidad**._


In [2]:

!git clone https://github.com/ju4n1t0x/eficiencia_energetica.git

Cloning into 'eficiencia_energetica'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 95 (delta 55), reused 55 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 1.36 MiB | 6.09 MiB/s, done.
Resolving deltas: 100% (55/55), done.


In [3]:
BASE_DIR = Path('/content/eficiencia_energetica')

BASE_DIR.exists()

True

## 0.3 Definición de rutas del proyecto

Una vez verificado el acceso a la carpeta principal, definimos las rutas que vamos a usar durante el trabajo.

Esto nos permite leer el dataset original, guardar datos procesados y exportar resultados sin escribir rutas manualmente en cada parte del notebook.


In [4]:
# Carpetas principales
DOCS_DIR = BASE_DIR / '00_documentacion'
NOTEBOOKS_DIR = BASE_DIR / '01_notebooks'
DATA_DIR = BASE_DIR / '02_data'
RAW_DATA_DIR = DATA_DIR / 'raw'
PROCESSED_DATA_DIR = DATA_DIR / 'processed'

OUTPUTS_DIR = BASE_DIR / '03_outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
TABLES_DIR = OUTPUTS_DIR / 'tables'
MODELS_DIR = OUTPUTS_DIR / 'models'

REPORTS_DIR = BASE_DIR / '04_reports'

## 0.4 Verificación de estructura de carpetas

Verificamos que las carpetas principales existan. En caso de que alguna carpeta de resultados todavía no esté creada, se crea automáticamente.


In [5]:
folders = [
    DOCS_DIR,
    NOTEBOOKS_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    MODELS_DIR,
    REPORTS_DIR
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)
    print(folder, "OK" if folder.exists() else "NO EXISTE")

/content/eficiencia_energetica/00_documentacion OK
/content/eficiencia_energetica/01_notebooks OK
/content/eficiencia_energetica/02_data/raw OK
/content/eficiencia_energetica/02_data/processed OK
/content/eficiencia_energetica/03_outputs/figures OK
/content/eficiencia_energetica/03_outputs/tables OK
/content/eficiencia_energetica/03_outputs/models OK
/content/eficiencia_energetica/04_reports OK


---


# 1. Comprensión del negocio




En esta primera fase se busca entender el problema desde una mirada general, antes de pasar al análisis de datos.

El objetivo es explicar qué problema se quiere abordar, por qué tiene sentido trabajarlo, quiénes podrían beneficiarse con una solución de este tipo y qué se espera lograr con el proyecto.

## 1.1 Contexto del problema

El consumo energético es una variable importante tanto para los usuarios como para las empresas distribuidoras y los organismos encargados de la planificación energética.

Cuando la demanda aumenta en ciertos períodos, puede ser necesario anticipar recursos, reforzar infraestructura o tomar decisiones operativas con mayor información.

Por este motivo, contar con un análisis de la demanda histórica puede ayudar a entender mejor el comportamiento del consumo y detectar momentos de mayor exigencia para el sistema.


## 1.2 Problema identificado

El problema que se busca abordar es la dificultad para anticipar períodos de mayor demanda energética a partir de datos históricos.

Si bien existen registros de consumo, estos datos por sí solos no siempre son fáciles de interpretar. Por eso se propone aplicar un proceso de análisis y minería de datos que permita encontrar patrones, comparar comportamientos y generar una base para posibles predicciones.


## 1.3 Stakeholders

Los principales actores relacionados con este problema son:

- Consumidores finales.
- Distribuidoras eléctricas.
- Gobierno u organismos reguladores.
- Equipos técnicos y operativos.
- Equipo desarrollador del proyecto.

Cada uno de estos actores puede beneficiarse de distintas formas. Los consumidores pueden verse favorecidos por una mejor planificación del servicio, las distribuidoras pueden anticipar períodos críticos y los organismos públicos pueden contar con información útil para la toma de decisiones.


## 1.4 Objetivo general

Analizar datos históricos de demanda energética para identificar patrones de consumo y evaluar la posibilidad de construir un modelo que permita estimar períodos de mayor demanda.


## 1.5 Objetivos específicos

- Analizar la evolución de la demanda energética a lo largo del tiempo.
- Identificar variaciones de consumo por año, mes o estación.
- Comparar el comportamiento de la demanda según las categorías disponibles en el dataset.
- Detectar posibles patrones o tendencias relevantes.
- Construir un modelo base que permita estimar la demanda energética.
- Evaluar los resultados obtenidos y sus limitaciones.


## 1.6 Alcance del proyecto

El alcance principal del trabajo será el análisis y modelado de la demanda energética a partir del dataset seleccionado.

El proyecto no busca predecir cortes eléctricos de forma directa, ya que para eso serían necesarios datos específicos sobre interrupciones del servicio. En cambio, se trabajará sobre la demanda energética como variable principal, entendiendo que su análisis puede servir como apoyo para la planificación y prevención de escenarios de alta exigencia.


## 1.7 Criterios de éxito del negocio

Desde el punto de vista del negocio, el proyecto será exitoso si permite:

- Comprender mejor el comportamiento histórico de la demanda energética.
- Identificar períodos de mayor consumo.
- Generar información útil para apoyar la toma de decisiones.
- Presentar los resultados de forma clara para usuarios técnicos y no técnicos.


## 1.8 Objetivo de minería de datos

El presente proyecto tiene como propósito traducir el problema de negocio identificado en las secciones anteriores en un objetivo técnico concreto y medible. Para ello, se define con precisión qué se quiere predecir, con qué datos, mediante qué tipo de tarea y bajo qué criterios se considerará que el modelo es exitoso.

**Variable objetivo**

La variable que el modelo intentará estimar es `demanda_MWh`, que representa el consumo mensual de energía eléctrica expresado en Megavatios-hora (MWh). Esta columna está disponible en el dataset de CAMMESA y contiene valores numéricos continuos correspondientes al consumo registrado por cada agente del mercado eléctrico mayorista en un período mensual determinado.

**Tipo de tarea**

Se trata de un problema de **regresión**, dado que la variable objetivo es numérica y continua. El objetivo no es clasificar períodos de consumo en categorías como "alto" o "bajo", sino estimar el valor cuantitativo de la demanda energética. Esta distinción es fundamental, ya que determina el tipo de algoritmo a utilizar y las métricas con las que se evaluará el modelo.

**Justificación de la elección**

La elección de `demanda_MWh` como variable objetivo responde directamente al problema de negocio planteado: la dificultad para anticipar períodos de mayor consumo energético. El dataset disponible cuenta con registros históricos mensuales desagregados por agente, región, provincia y categoría tarifaria, lo que permite construir variables predictoras relevantes a partir de información temporal y geográfica. Esta combinación de variables hace que el dataset sea adecuado para abordar el problema mediante un modelo de regresión supervisada.

**Criterio técnico de éxito**

El modelo será evaluado mediante las siguientes métricas estándar para problemas de regresión:

| Métrica | Descripción | Valor esperado |
|--------|-------------|----------------|
| MAE (Error Absoluto Medio) | Mide el error promedio en las mismas unidades que la variable objetivo (MWh) | Lo más bajo posible en relación a la escala del consumo |
| RMSE (Raíz del Error Cuadrático Medio) | Similar al MAE pero penaliza más los errores grandes | Lo más bajo posible |
| R² (Coeficiente de Determinación) | Indica qué proporción de la variación de los datos explica el modelo | Superior a 0.7 |

Se considerará que el modelo tiene un desempeño aceptable si logra un R² superior a 0.7 y si los valores de MAE y RMSE son razonables en relación a la magnitud típica de la demanda registrada en el dataset.

**Alineación con el modelado**

Este objetivo técnico es el punto de partida directo para la fase de modelado desarrollada en la sección 5. Allí se seleccionarán las variables predictoras, se dividirán los datos en conjuntos de entrenamiento y prueba, se entrenará un modelo base de regresión y se generarán las primeras predicciones de demanda energética. La evaluación de dichas predicciones se realizará en la sección 6, utilizando las métricas definidas en este punto.

**Alcance y limitaciones del objetivo**

Es importante aclarar que el foco de este proyecto es la **demanda energética** y no los cortes eléctricos. Si bien ambos fenómenos están relacionados, predecir interrupciones del servicio requeriría datos específicos sobre fallas, mantenimientos e incidentes que no forman parte del dataset disponible. Por este motivo, el análisis se centra en estimar el consumo como variable proxy para identificar períodos de alta exigencia sobre el sistema eléctrico, información que resulta igualmente valiosa para la toma de decisiones operativas y de planificación.

---


# 2. Comprensión de los datos




En esta fase se realiza el primer acercamiento al dataset.

La idea es conocer de dónde provienen los datos, qué información contienen, qué columnas tienen, qué período cubren y qué problemas podrían aparecer antes de avanzar con la limpieza y el modelado.

## 2.1 Fuente de datos



El dataset utilizado corresponde a registros históricos de demanda energética.

En esta sección se documentará la fuente del archivo, el nombre del dataset, el período cubierto, la unidad de medida y las principales variables disponibles.

El dataset obtenido proviene de CAMMESA (Compañía Administradora del Mercado Mayorista Eléctrico S.A.) es la empresa privada sin fines de lucro, con participación pública, responsable de administrar y operar el Mercado Eléctrico Mayorista (MEM) en Argentina. Su función principal es coordinar la generación, transporte y demanda de energía, equilibrando el suministro eléctrico.

Trabajaremos con el archivo csv nombrado como ***demanda-ltimos-aos*** obtenido de la siguiente URL: http://datos.energia.gob.ar/dataset/publicaciones-cammesa/archivo/ae008cdf-ed5d-4a85-90ee-5f4c53704e79.

Algunos de los datos relevantes que queremos destacar son los siguientes:

* Período cubierto: enero 2017 — diciembre 2020 (4 años completos)
* Unidad de medida: Megavatios-hora (MWh) por mes, por agente
* Nivel de detalle: Mensual, desagregado por agente, región, provincia, tipo de agente y tarifa. Cada fila representa el consumo mensual de un agente específico del mercado eléctrico.📅 Período cubierto: enero 2017 — diciembre 2020 (4 años completos)
* Unidad de medida: Megavatios-hora (MWh) por mes, por agente
* Nivel de detalle: Mensual, desagregado por agente, región, provincia, tipo de agente y tarifa. Cada fila representa el consumo mensual de un agente específico del mercado eléctrico.

Y como así mismo, también encontramos como columnas relevantes las siguientes:

| Columna              | Descripción                                                                 | 
|--------------------|-----------------------------------------------------------------------------|
| anio / mes         | Período de medición                                                    |
| region / provincia | Ubicación geográfica del consumo                                                    |
| demanda_MWh        | Consumo mensual en MWh ← variable objetivo                                                    |
| categoría_area / categoría_consumo | Como identificadores del tipo de consumidor y área                                   |

Haciendo un repaso por las variables, el objetivo al que queremos llegar y los datos con los que trabajaremos creemos que algunas de las limitaciones que tendremos son las siguientes:

* El período disponible abarca solo 2017–2020, lo que limita la detección de tendencias de largo plazo.
* No incluye datos de cortes o interrupciones del servicio.
* La columna fecha_proceso corresponde a la fecha de carga al sistema, no a la fecha de consumo real.
* El dataset no tiene valores nulos (40.388 registros completos), pero requiere revisión de outliers en demanda_MWh.


## 2.2 Carga inicial del dataset




En esta sección cargamos el archivo original desde la carpeta correspondiente del proyecto.

El dataset original se mantiene sin modificaciones dentro de la carpeta de datos crudos, y cualquier transformación posterior se guardará como una nueva versión procesada.

In [6]:
csv_files = list(RAW_DATA_DIR.glob('*.csv'))
csv_files

[PosixPath('/content/eficiencia_energetica/02_data/raw/demanda-ltimos-aos.csv')]

Seleccionamos el archivo encontrado para cargarlo en el notebook.


In [7]:
csv_path = csv_files[0]
csv_path

PosixPath('/content/eficiencia_energetica/02_data/raw/demanda-ltimos-aos.csv')

Cargamos el dataset en un DataFrame y mostramos las primeras filas para verificar la lectura.


In [8]:
df = pd.read_csv(csv_path)


## 2.3 Descripción inicial de los datos



En esta parte se revisarán las primeras filas del dataset, la cantidad de registros, la cantidad de columnas y los tipos de datos disponibles.

Esto permitirá tener una primera idea de la estructura general del archivo antes de aplicar limpieza o transformaciones.


In [9]:
#visualizar las primeras filas del dataframe
df.head(10)

,id,anio,mes,agente_nemo,agente_descripcion,tipo_agente,region,provincia,categoria_area,categoria_demanda,tarifa,categoria_tarifa,demanda_MWh,fecha_proceso,lote_id_log,indice_tiempo
0,699232,2017,1,AARGTAOY,AEROP ARG 2000 - Aeroparque,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,1990.439,2020-05-05 11:06:49.000713,67,2017-01
1,699233,2017,1,ABRILHCY,ABRIL CLUB DE CAMPO,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,1609.464,2020-05-05 11:06:49.000713,67,2017-01
2,699234,2017,1,ACARQQ3Y,ASOC.COOP.ARG. - Quequén,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,421.334,2020-05-05 11:06:49.000713,67,2017-01
3,699235,2017,1,ACARSLSY,ASOC.COOP.ARG. - San Lorenzo,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,449.145,2020-05-05 11:06:49.000713,67,2017-01
4,699236,2017,1,ACERBR1Y,Planta Bragado,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,18073.170,2020-05-05 11:06:49.000713,67,2017-01
5,699237,2017,1,ACINROSY,ACINDAR ROSARIO EX-NAVARRO,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,157.891,2020-05-05 11:06:49.000713,67,2017-01
6,699238,2017,1,ACINTBOY,ACINDAR PTA. TABLADA,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,2017.775,2020-05-05 11:06:49.000713,67,2017-01
7,699239,2017,1,ACINVCSZ,ACINDAR PTA. V. CONSTITUCION,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,56074.663,2020-05-05 11:06:49.000713,67,2017-01
8,699240,2017,1,ACUYLCMY,ACEROS CUYANOS S.A.- L.DE CUYO,GU,CUYO,MENDOZA,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,586.065,2020-05-05 11:06:49.000713,67,2017-01
9,699241,2017,1,AEEZECCY,AEROP ARG 2000 EX FAA EZEIZA,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,4631.846,2020-05-05 11:06:49.000713,67,2017-01


In [10]:
#contar el número de filas y columnas del dataframe
df.shape

(40388, 16)

***El data set cuenta con un total de 40.388 filas y un total de 16 columnas por cada fila***

In [11]:
#vemos que tipo de datos posee el dataframe
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40388 entries, 0 to 40387
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  40388 non-null  int64  
 1   anio                40388 non-null  int64  
 2   mes                 40388 non-null  int64  
 3   agente_nemo         40388 non-null  object 
 4   agente_descripcion  40388 non-null  object 
 5   tipo_agente         40388 non-null  object 
 6   region              40388 non-null  object 
 7   provincia           40388 non-null  object 
 8   categoria_area      40388 non-null  object 
 9   categoria_demanda   40388 non-null  object 
 10  tarifa              40388 non-null  object 
 11  categoria_tarifa    40388 non-null  object 
 12  demanda_MWh         40388 non-null  float64
 13  fecha_proceso       40388 non-null  object 
 14  lote_id_log         40388 non-null  int64  
 15  indice_tiempo       40388 non-null  object 
dtypes: f

In [12]:
# y sacamos los primeros datos descriptivos para ver rangos/outliers
df.describe()

,id,anio,mes,demanda_MWh,lote_id_log
count,40388.000000,40388.000000,40388.000000,4.038800e+04,40388.0
mean,719425.500000,2017.955903,6.279712,1.033144e+04,67.0
std,11659.155673,0.871699,3.527905,4.594102e+04,0.0
min,699232.000000,2017.000000,1.000000,-6.554332e+03,67.0
25%,709328.750000,2017.000000,3.000000,2.754673e+02,67.0
50%,719425.500000,2018.000000,6.000000,1.207639e+03,67.0
75%,729522.250000,2019.000000,9.000000,4.244417e+03,67.0
max,739619.000000,2020.000000,12.000000,1.572308e+06,67.0


## 2.4 Diccionario de datos




Se armará un diccionario simple con las columnas principales del dataset.

El objetivo es explicar qué significa cada variable, qué tipo de dato contiene y si será utilizada para el análisis exploratorio o para el modelo.

| Campo              | Descripción                                                                 | Tipo     | Analisis exploratorio o modelo        |
|--------------------|-----------------------------------------------------------------------------|----------|---------------------------------------|
| id                 | Identificador de cliente                                                    | int64    | Clave primaria potencial              |
| anio               | Año en que se realiza la medición                                           | int64    | Formato YYYY                          |
| mes                | Mes en que se realiza la medición                                           | int64    | Valores de 1 a 12                     |
| agente_nemo        | Identificador de agente                                                     | object   | Código único del agente               |
| agente_descripcion | Nombre del agente                                                           | object   | Descriptivo                           |
| tipo_agente        | Clasificador del tipo de agente                                             | object   | Ej: Distribuidor / Generador          |
| region             | Región geográfica a la que pertenece                                        | object   | Nivel macro                           |
| provincia          | Provincia a la que pertenece                                                | object   | Nivel más granular                    |
| categoria_area     | Distingue al agente por consumo (Gran Usuario o Distribuidor)               | object   | Clasificación de consumo              |
| tarifa             | Determina el monto de tarifa según el tipo de agente                        | object   | Puede ser categórica                  |
| categoria_tarifa   | Agrupa las tarifas según condición del agente                               | object   | Nivel de agrupación                   |
| demanta_MWh        | Consumo mensual de energía en megawatts/hora                                | float64  | Variable numérica continua            |
| fecha_proceso      | Fecha en la que se realizó la medición del consumo                          | object   | Conviene convertir a datetime         |
| lote_id_log        | Lote al que pertenece la medición                                           | int64    | Puede servir para trazabilidad        |
| indice_tiempo      | Índice temporal                                                             | object   | No definido (revisar significado)     |

## 2.5 Verificación inicial de calidad




En esta sección se revisará si existen valores nulos, duplicados, fechas inconsistentes, valores negativos o categorías mal registradas.

Esta revisión es importante para decidir qué transformaciones serán necesarias en la fase de preparación de datos.

In [13]:
#verificamos si hay datos nulos dentro del dataframe
datos_nulos = df.isnull().sum()
print("Datos nulos por columna:")
print(datos_nulos)
print("\nTotal de datos nulos en el dataframe:", datos_nulos.sum())

Datos nulos por columna:
id                    0
anio                  0
mes                   0
agente_nemo           0
agente_descripcion    0
tipo_agente           0
region                0
provincia             0
categoria_area        0
categoria_demanda     0
tarifa                0
categoria_tarifa      0
demanda_MWh           0
fecha_proceso         0
lote_id_log           0
indice_tiempo         0
dtype: int64

Total de datos nulos en el dataframe: 0


In [14]:
#identificamos valores unicos
df.nunique()

,0
id,40388
anio,4
mes,12
agente_nemo,564
agente_descripcion,560
tipo_agente,4
region,9
provincia,22
categoria_area,11
categoria_demanda,2


---


En base a los valores unicos podemos ver que:
- el dataframe cuenta efectivamente con 40.388 registros
- que contiene mediciones de 4 años diferentes
- que a su vez esta dividido en 12 meses
- que contamos con 564 agentes
- pero que solo tenemos 560 descripciones, por lo cual tiene que haber algun agente duplicado con diferente numero de agente_nemo
- que contamos con 4 tipos de agentes diferentes
- pertenecientes a 9 regiones
- de un total de 22 provincias relevadas
- que tambien tenemos 11 areas
- 2 categorias de demanda diferentes
- hay 21 tipo de tarifas
- 4 categorias para agrupar las tarifas
- que la demanda_MWh posee una cardinalidad alta
- que todos los registros fueron procesados en la misma fecha y con el mismo lote
- el indice de tiempo tampoco es consistente, ya que si tenemos 4 años y 12 meses deberia dar un total de 48 indices de tiempo 

In [15]:
#contamos si hay algun dato duplicado dentro del dataframe
duplicados = df.duplicated().sum()
print(f'El dataframe posee {duplicados} valores duplicados')

El dataframe posee 0 valores duplicados


In [16]:
#verificamos la inconsistencia entre la cantidad de datos en agente_nemo y la cantidad de datos en agente_descripcion
agente_nemo_unicos = df.groupby('agente_nemo')['agente_descripcion'].nunique()
print(agente_nemo_unicos)
agente_descripcion_unicos = df.groupby('agente_descripcion')['agente_nemo'].nunique().sort_values(ascending=False)
print(agente_descripcion_unicos)

agente_nemo
AARGTAOY    1
ABRILHCY    1
ACARQQ3Y    1
ACARSLSY    1
ACERBR1Y    1
           ..
YPFSECZY    1
YPFTORUZ    1
YPFTREUA    1
YPFTREUZ    1
ZUCARACY    1
Name: agente_descripcion, Length: 564, dtype: int64
agente_descripcion
CERVECERIA Y MALTERIA QUILMES     2
YPF S.A.                          2
LOMA NEGRA - PTA. OLAVARRIA       2
PETROQUIMICA CUYO - L. Cuyo       2
MONDELEZ ARGENTINA                1
                                 ..
COOP.ELECT.DE BARILOCHE           1
COOPER.ELEC.GODOY CRUZ DISTRIB    1
COOPERATIVA DE BARKER             1
COOPERATIVA DE LEZAMA             1
COOP. VILLA GESELL                1
Name: agente_nemo, Length: 560, dtype: int64


Podemos ver que una misma descripcion aparece con mas de un codigo de agente_nemo diferente:
Cerveceria y Malteria Quilmes
YPF S.A.
LOMA NEGRA - PTA. OLAVARRIA
PETROQUIMICA CUYO - L. CUYO
***por lo cual la variable agente_nemo no es un identificador logico de una entidad, ya que mismas entidades poseen codigos diferentes, puede deberse a distintos puntos de consumo de un mismo agente***

In [17]:
#verificamos los faltantes temporales
df[['anio','mes']].drop_duplicates().sort_values(by=['anio','mes'])

,anio,mes
0,2017,1
970,2017,2
1663,2017,3
3176,2017,4
4371,2017,5
5983,2017,6
7142,2017,7
8476,2017,8
9496,2017,9
11003,2017,10


**Podemos ver que en el anio 2020 solo contamos con el registro de 2 meses (enero y febrero)**

In [18]:
#vemos cuales son todas las categorias con las que contamos en el dataframe
df['categoria_area'].value_counts()




,count
categoria_area,
Gran Usuario MEM,18670
Resto,17441
Mendoza,1009
Entre Rios,957
Edesur,336
Tucuman,335
Eden,335
Edeaba,334
Santa Fe,331


Dentro del dataframe nos podemos encontrar con las siguientes categorias de area:
- Gran Usuario MEM - 18670 registros
- Resto - 17441 registros
- Mendoza - 1009 registros
- Entre Rios - 957 registros
- Edesur - 336 registros
- Tucuman - 335 registros
- Eden - 335 registros
- Edeaba - 334 registros
- Santa Fe - 331 registros
- Edenor - 327 registros 
- Cordoba - 313 registros

In [19]:
#vemos las categorias de demanda
df['categoria_demanda'].value_counts()



,count
categoria_demanda,
Distribuidor,21718
Gran Usuario,18670


Dentro del dataframe nos podemos encontrar con las siguientes categorias de demanda:
- Distribuidor - 21718 registros
- Gran Usuario - 18670 registros
        

In [ ]:
#vemos las categorias de tarifa
df['categoria_tarifa'].value_counts()

,count
categoria_tarifa,
Industrial/Comercial Grande,21144
Residencial,15628
Comercial,2850
Mercado a Término,766


Dentro del dataframe nos podemos encontrar con las siguientes categorias de tarifa:
- Industrial/Coimercial Grande - 21144 registros
- Residencial - 15628 registros
- Comercial 2850 registros
- Mercado a Termino - 766 registros

In [21]:
df.columns = df.columns.str.lower()
anios = sorted(df['anio'].unique())

print("\n--- COLUMNAS QUE NO APORTAN ---")
columnas_no_aportan = []
for col in df.columns:
    if df[col].nunique() <= 1:
        columnas_no_aportan.append(col)
        print(f"La columna '{col}' es constante (solo tiene el valor: {df[col].iloc[0]})")


ids_tecnicos = ['id', 'lote_id'] 
print(f"Sugerencia de eliminación (IDs técnicos): {ids_tecnicos}")

print("\n--- DATOS PARA LA CONCLUSIÓN ---")
print(f"El dataset cubre un periodo de {len(anios)} años.")
print(f"La demanda promedio general es de: {df['demanda_mwh'].mean():.2f} MWh")


--- COLUMNAS QUE NO APORTAN ---
La columna 'fecha_proceso' es constante (solo tiene el valor: 2020-05-05 11:06:49.000713)
La columna 'lote_id_log' es constante (solo tiene el valor: 67)
Sugerencia de eliminación (IDs técnicos): ['id', 'lote_id']

--- DATOS PARA LA CONCLUSIÓN ---
El dataset cubre un periodo de 4 años.
La demanda promedio general es de: 10331.44 MWh


## Conclusion



Tras el procesamiento inicial y la auditoría de calidad del dataset de eficiencia energética, se han extraído las siguientes observaciones clave:

* **Columnas Redundantes:** Se han identificado variables con varianza cero (constantes) que no aportan valor predictivo ni informativo al modelo, específicamente `fecha_proceso` y `lote_id_log`. Se recomienda su eliminación para simplificar el dataframe.
* **Depuración de IDs:** Se sugiere la remoción de identificadores técnicos como `id` y `lote_id`, dado que son registros administrativos que no guardan relación con el comportamiento del consumo energético.

* **Cobertura Temporal:** El estudio abarca un periodo sólido de **4 años**, lo cual es estadísticamente significativo para observar patrones estacionales y tendencias de consumo a largo plazo.
* **Métrica de Demanda:** La demanda promedio general se sitúa en **10,331.44 MWh**. Este valor servirá como línea base (*baseline*) para comparar las fluctuaciones según diferentes variables externas o periodos específicos.

Con el dataset limpio de ruido técnico y columnas constantes, el siguiente paso consistirá en el análisis de la distribución de la variable `demanda_mwh` y su correlación con factores temporales o climáticos, avanzando así en la etapa de comprensión de los datos dentro del marco CRISP-DM.

# 3. Preparación de los datos




En esta fase se preparará el dataset para que pueda ser utilizado en el análisis exploratorio y en el modelado.

La preparación puede incluir selección de columnas, limpieza de valores, corrección de formatos, creación de nuevas variables y guardado de una versión limpia del dataset.

## 3.1 Selección de datos



Se seleccionarán las columnas necesarias para responder los objetivos del proyecto.

También se identificarán columnas que no aporten al análisis o que no puedan utilizarse por problemas de calidad, formato o falta de relación con el objetivo.

In [36]:
df.head(10)

,id,anio,mes,agente_nemo,agente_descripcion,tipo_agente,region,provincia,categoria_area,categoria_demanda,tarifa,categoria_tarifa,demanda_mwh,fecha_proceso,lote_id_log,indice_tiempo
0,699232,2017,1,AARGTAOY,AEROP ARG 2000 - Aeroparque,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,1990.439,2020-05-05 11:06:49.000713,67,2017-01
1,699233,2017,1,ABRILHCY,ABRIL CLUB DE CAMPO,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,1609.464,2020-05-05 11:06:49.000713,67,2017-01
2,699234,2017,1,ACARQQ3Y,ASOC.COOP.ARG. - Quequén,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,421.334,2020-05-05 11:06:49.000713,67,2017-01
3,699235,2017,1,ACARSLSY,ASOC.COOP.ARG. - San Lorenzo,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,449.145,2020-05-05 11:06:49.000713,67,2017-01
4,699236,2017,1,ACERBR1Y,Planta Bragado,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,18073.170,2020-05-05 11:06:49.000713,67,2017-01
5,699237,2017,1,ACINROSY,ACINDAR ROSARIO EX-NAVARRO,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,157.891,2020-05-05 11:06:49.000713,67,2017-01
6,699238,2017,1,ACINTBOY,ACINDAR PTA. TABLADA,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,2017.775,2020-05-05 11:06:49.000713,67,2017-01
7,699239,2017,1,ACINVCSZ,ACINDAR PTA. V. CONSTITUCION,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,56074.663,2020-05-05 11:06:49.000713,67,2017-01
8,699240,2017,1,ACUYLCMY,ACEROS CUYANOS S.A.- L.DE CUYO,GU,CUYO,MENDOZA,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,586.065,2020-05-05 11:06:49.000713,67,2017-01
9,699241,2017,1,AEEZECCY,AEROP ARG 2000 EX FAA EZEIZA,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,4631.846,2020-05-05 11:06:49.000713,67,2017-01


### Hacemos la busqueda de columnas y designamos las que "Visualmente" deberiamos borrar:

In [23]:
# Normalizamos las columnas
df.columns = df.columns.str.strip().str.lower()

# Definimos las columnas que son innecesarias a simple vista
columnas_innecesarias = ['id', 'lote_id_log', 'fecha_proceso']

# Detectamos columnas que solo tengan contenido repetido que sea igual o menor a 1
columnas_constantes = [col for col in df.columns if df[col].nunique() <= 1]

# Efectuamos la limpieza
columnas_finales_a_eliminar = list(set(columnas_innecesarias + columnas_constantes))
df_limpio = df.drop(columns=[col for col in columnas_finales_a_eliminar if col in df.columns])


# Salida del print con columnas eliminadas automaticamente y declaradas.
print("LIMPIEZA AUTOMÁTICA DE COLUMNAS")
print(f"Columnas constantes detectadas: {columnas_constantes}")
print(f"Columnas técnicas eliminadas: {columnas_innecesarias}")
print(f"\nQuedan solo las columnas: {df_limpio.columns.tolist()}")

display(df_limpio.head())

LIMPIEZA AUTOMÁTICA DE COLUMNAS
Columnas constantes detectadas: ['fecha_proceso', 'lote_id_log']
Columnas técnicas eliminadas: ['id', 'lote_id_log', 'fecha_proceso']

Quedan solo las columnas: ['anio', 'mes', 'agente_nemo', 'agente_descripcion', 'tipo_agente', 'region', 'provincia', 'categoria_area', 'categoria_demanda', 'tarifa', 'categoria_tarifa', 'demanda_mwh', 'indice_tiempo']


,anio,mes,agente_nemo,agente_descripcion,tipo_agente,region,provincia,categoria_area,categoria_demanda,tarifa,categoria_tarifa,demanda_mwh,indice_tiempo
0,2017,1,AARGTAOY,AEROP ARG 2000 - Aeroparque,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,1990.439,2017-01
1,2017,1,ABRILHCY,ABRIL CLUB DE CAMPO,GU,GRAN BS.AS.,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,1609.464,2017-01
2,2017,1,ACARQQ3Y,ASOC.COOP.ARG. - Quequén,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,421.334,2017-01
3,2017,1,ACARSLSY,ASOC.COOP.ARG. - San Lorenzo,GU,LITORAL,SANTA FE,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,449.145,2017-01
4,2017,1,ACERBR1Y,Planta Bragado,GU,BUENOS AIRES,BUENOS AIRES,Gran Usuario MEM,Gran Usuario,GUMAS/AUTOGENERADORES,Industrial/Comercial Grande,18073.170,2017-01


## 3.2 Limpieza de datos




Se aplicarán las correcciones necesarias sobre el dataset.

Esto puede incluir tratamiento de nulos, eliminación de duplicados, corrección de nombres de columnas, revisión de categorías y control de valores fuera de rango.

### Vamos a buscar las filas que estén repetidas:

In [ ]:
num_duplicados = df_limpio.duplicated().sum()
if num_duplicados > 0:
    df_limpio.drop_duplicates(inplace=True)
    print(f"Se eliminaron {num_duplicados} filas duplicadas exactas.")
else:
    print("No se encontraron filas duplicadas.")

En esta celda lo que hicimos fue visualizar que no haya dentro del dataset columnas repetidas que puedan sesgar el analisis

### Buscamos las columnas con datos nulos:

In [ ]:
# Analizamos las columnas, vemos si hay nulos y detallamos las que tienen datos nulos.
## en caso de no tener nulos arrojamos un print.


nulos_por_columna = df_limpio.isnull().sum()
columnas_con_nulos = nulos_por_columna[nulos_por_columna > 0]

# Armamos el IF en caso de no tener nulo.
if not columnas_con_nulos.empty:
    info_limpieza = pd.DataFrame({
        'Tipo': df_limpio[columnas_con_nulos.index].dtypes,
        'Nulos': columnas_con_nulos,
        '% Nulos': (columnas_con_nulos / len(df_limpio)) * 100
    })

# Arrojamos el Print
    print("DIAGNÓSTICO DE DATOS FALTANTES")
    display(info_limpieza.sort_values(by='% Nulos', ascending=False))
else:
    print("No existen datos nulos en ninguna columna del dataset.")

En esta celda, nos centramos en la obtencion de datos nulos. Pudimos verificar que el dataset elegido no cuenta con datos nulos, por lo cual no debemos realizar un tratamiento de ellos.

### Noramlizacion de las columnas STR

In [ ]:
columnas_upper = [
    'agente_nemo',
    'ag'
    'tipo_agente',
    'region',
    'provincia',
    'categoria_area',
    'categoria_demanda',
    'tarifa',
    'categoria_tarifa',

]

for col in columnas_upper:
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio[col].str.upper()
    else:
        print(f"Columna '{col}' no encontrada para estandarización.")

df_limpio

,anio,mes,agente_nemo,agente_descripcion,tipo_agente,region,provincia,categoria_area,categoria_demanda,tarifa,categoria_tarifa,demanda_mwh,indice_tiempo
0,2017,1,AARGTAOY,AEROP ARG 2000 - AEROPARQUE,GU,GRAN BS.AS.,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS/AUTOGENERADORES,INDUSTRIAL/COMERCIAL GRANDE,1990.439,2017-01
1,2017,1,ABRILHCY,ABRIL CLUB DE CAMPO,GU,GRAN BS.AS.,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS/AUTOGENERADORES,INDUSTRIAL/COMERCIAL GRANDE,1609.464,2017-01
2,2017,1,ACARQQ3Y,ASOC.COOP.ARG. - QUEQUÉN,GU,BUENOS AIRES,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS/AUTOGENERADORES,INDUSTRIAL/COMERCIAL GRANDE,421.334,2017-01
3,2017,1,ACARSLSY,ASOC.COOP.ARG. - SAN LORENZO,GU,LITORAL,SANTA FE,GRAN USUARIO MEM,GRAN USUARIO,GUMAS/AUTOGENERADORES,INDUSTRIAL/COMERCIAL GRANDE,449.145,2017-01
4,2017,1,ACERBR1Y,PLANTA BRAGADO,GU,BUENOS AIRES,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS/AUTOGENERADORES,INDUSTRIAL/COMERCIAL GRANDE,18073.170,2017-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...
40383,2020,2,MALTPU2Y,MALTERIA PAMPA S.A.,GU,BUENOS AIRES,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS/AUTOGENERADORES,INDUSTRIAL/COMERCIAL GRANDE,996.681,2020-02
40384,2020,2,CAPECOUZ,CAPEX S.A.-BELLAVISTA,GU,PATAGONICA,CHUBUT,GRAN USUARIO MEM,GRAN USUARIO,GUMAS/AUTOGENERADORES,INDUSTRIAL/COMERCIAL GRANDE,1904.777,2020-02
40385,2020,2,ALSOALOY,ALIMENTOS DE SOJA S.A.,GU,GRAN BS.AS.,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS/AUTOGENERADORES,INDUSTRIAL/COMERCIAL GRANDE,805.041,2020-02
40386,2020,2,SAIGCA1Y,SAINT GOBAIN ARG. S.A.,GU,BUENOS AIRES,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS/AUTOGENERADORES,INDUSTRIAL/COMERCIAL GRANDE,0.000,2020-02


En esta celda convertimos todas las columnas de tipo str a mayusculas para un mejor tratamiento de los datos

In [ ]:
for col in columnas_upper:
    if col in df_limpio.columns:
        df_limpio[col] = (
            df_limpio[col]
            .astype(str)
            .str.normalize('NFKD')  # Normaliza caracteres acentuados
            .str.encode('ascii', errors='ignore')  # Elimina caracteres no ASCII
            .str.decode('utf-8')  # Decodifica de nuevo a UTF-
            .str.strip()
            .str.replace(r'[\.\-\/]', ' ', regex=True)  # elimina ., -, /
            .str.replace(r'\s+', ' ', regex=True)       # colapsa espacios
        )

df_limpio.head(10)


,anio,mes,agente_nemo,agente_descripcion,tipo_agente,region,provincia,categoria_area,categoria_demanda,tarifa,categoria_tarifa,demanda_mwh,indice_tiempo
0,2017,1,AARGTAOY,AEROP ARG 2000 AEROPARQUE,GU,GRAN BS AS,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,1990.439,2017-01
1,2017,1,ABRILHCY,ABRIL CLUB DE CAMPO,GU,GRAN BS AS,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,1609.464,2017-01
2,2017,1,ACARQQ3Y,ASOC COOP ARG QUEQUEN,GU,BUENOS AIRES,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,421.334,2017-01
3,2017,1,ACARSLSY,ASOC COOP ARG SAN LORENZO,GU,LITORAL,SANTA FE,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,449.145,2017-01
4,2017,1,ACERBR1Y,PLANTA BRAGADO,GU,BUENOS AIRES,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,18073.170,2017-01
5,2017,1,ACINROSY,ACINDAR ROSARIO EX NAVARRO,GU,LITORAL,SANTA FE,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,157.891,2017-01
6,2017,1,ACINTBOY,ACINDAR PTA TABLADA,GU,GRAN BS AS,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,2017.775,2017-01
7,2017,1,ACINVCSZ,ACINDAR PTA V CONSTITUCION,GU,LITORAL,SANTA FE,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,56074.663,2017-01
8,2017,1,ACUYLCMY,ACEROS CUYANOS S A L DE CUYO,GU,CUYO,MENDOZA,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,586.065,2017-01
9,2017,1,AEEZECCY,AEROP ARG 2000 EX FAA EZEIZA,GU,GRAN BS AS,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,4631.846,2017-01


Si bien el dataframe no tenia datos nulos, nos encontramos con que muchas columnas utilizaban espacios no estandarizados, signos como tildes, guiones medio y barras invertidas, por lo cual decidimos eliminarlas, quitar todos los tildes y signos que no fueran un caracter alfabetico, y estandarizar el los espacios a solo uno entre cada palabra 1 en todas las columnas

## 3.3 Construcción de nuevas variables



A partir de las columnas existentes se podrán crear nuevas variables útiles para el análisis.

Por ejemplo, si el dataset contiene fechas, se podrán crear columnas de año, mes o estación para estudiar el comportamiento temporal de la demanda.

In [ ]:
#definimos una funcion para obtener la estacion del año a partir del mes en el que se realiza la medicion
def obtener_estacion(mes):
    mes = mes % 12
    if mes in [12, 1, 2]:
        return 'VERANO'
    elif mes in [3, 4, 5]:
        return 'OTOÑO'
    elif mes in [6, 7, 8]:
        return 'INVIERNO'
    else:
        return 'PRIMAVERA'
    
df_limpio['estacion'] = df_limpio['mes'].apply(obtener_estacion)

#Hacemos un conteo de la cantidad de registros por estación para verificar que se hayan asignado correctamente
total_estaciones = df_limpio['estacion'].value_counts()
print("Total de registros por estación:")
print(total_estaciones)


Total de registros por estación:
estacion
PRIMAVERA    12883
INVIERNO      9694
OTOÑO         9603
VERANO        8208
Name: count, dtype: int64


Creamos la columna estacion para permitir el analisis segun la estacion del a;o en la que se realiza la medicion

In [46]:
total_consumo_por_region = df_limpio.groupby('region')['demanda_mwh'].sum().sort_values(ascending=False).round(2)
print("Total de consumo por región (mostrado en millones de GWh):")
print((total_consumo_por_region / 1_000_000).round(2))

Total de consumo por región (mostrado en millones de GWh):
region
GRAN BS AS      157.21
LITORAL          50.85
BUENOS AIRES     47.74
CENTRO           36.22
NOROESTE         34.79
NORESTE          30.38
CUYO             26.00
PATAGONICA       18.08
COMAHUE          16.00
Name: demanda_mwh, dtype: float64


Creamos la variable total_consumo_por_region para poder visualizar en millones los totales consumos por cada region

In [45]:
total_consumo_por_provincia = df_limpio.groupby('provincia')['demanda_mwh'].sum().sort_values(ascending=False).round(2)
print("Total de consumo por provincia (mostrado en millones de GWh):")
print((total_consumo_por_provincia / 1_000_000).round(2))

Total de consumo por provincia (mostrado en millones de GWh):
provincia
BUENOS AIRES      204.95
SANTA FE           39.25
CORDOBA            31.11
MENDOZA            18.87
CHUBUT             14.34
ENTRE RIOS         11.61
TUCUMAN            10.00
CHACO               9.27
CORRIENTES          9.22
MISIONES            7.53
SAN JUAN            7.12
NEUQUEN             6.99
SALTA               6.49
RIO NEGRO           6.18
CATAMARCA           5.18
SGO DEL ESTERO      5.16
SAN LUIS            5.11
LA RIOJA            4.59
FORMOSA             4.35
SANTA CRUZ          3.74
JUJUY               3.38
LA PAMPA            2.83
Name: demanda_mwh, dtype: float64


Creamos la variable total_consumo_por_provincia para verificar y contabilizar en millones, como varia el consumo entre las provincias.

In [47]:
total_consumo_por_categoria_area = df_limpio.groupby('categoria_area')['demanda_mwh'].sum().sort_values(ascending=False).round(2)
print("Total de consumo por categoría de área (mostrado en millones de GWh):")
print((total_consumo_por_categoria_area / 1_000_000).round(2))

Total de consumo por categoría de área (mostrado en millones de GWh):
categoria_area
RESTO               104.21
GRAN USUARIO MEM     76.79
EDENOR               69.39
EDESUR               55.43
SANTA FE             30.94
CORDOBA              29.03
MENDOZA              14.04
ENTRE RIOS           10.77
EDEN                  9.20
TUCUMAN               8.85
EDEABA                8.62
Name: demanda_mwh, dtype: float64


Creamos la variable total_consumo_por_categoria_area para poder contabilizar en millones y verificar como varia el consumo segun el area

In [48]:
total_consumo_por_categoria_demanda = df_limpio.groupby('categoria_demanda')['demanda_mwh'].sum().sort_values(ascending=False).round(2)
print("Total de consumo por categoría de demanda (mostrado en millones de GWh):")
print((total_consumo_por_categoria_demanda / 1_000_000).round(2))

Total de consumo por categoría de demanda (mostrado en millones de GWh):
categoria_demanda
DISTRIBUIDOR    340.47
GRAN USUARIO     76.79
Name: demanda_mwh, dtype: float64


Creamos la variable total_consumo_por_categoria_demanda para poder contabilizar en millones y verificar como varia el consumo segun la categoria de demanda

In [49]:
total_consumo_por_categoria_tarifa = df_limpio.groupby('categoria_tarifa')['demanda_mwh'].sum().sort_values(ascending=False).round(2)
print("Total de consumo por categoría de tarifa (mostrado en millones de GWh):")
print((total_consumo_por_categoria_tarifa / 1_000_000).round(2))

Total de consumo por categoría de tarifa (mostrado en millones de GWh):
categoria_tarifa
RESIDENCIAL                    178.74
COMERCIAL                      120.22
INDUSTRIAL COMERCIAL GRANDE    118.32
MERCADO A TERMINO               -0.01
Name: demanda_mwh, dtype: float64


Creamos la variable total_consumo_por_categoria_tarifa para poder contabilizar en millones y verificar como varia el consumo segun la categoria de demanda

## 3.4 Formateo final del dataset



Se revisará que el dataset quede en un formato adecuado para los gráficos y el modelo.

La versión final procesada se guardará en la carpeta de datos limpios para reutilizarla en las siguientes etapas del trabajo.

#### Guardamos el archivo como con principio:

In [50]:
df_limpio.head()

,anio,mes,agente_nemo,agente_descripcion,tipo_agente,region,provincia,categoria_area,categoria_demanda,tarifa,categoria_tarifa,demanda_mwh,indice_tiempo,estacion
0,2017,1,AARGTAOY,AEROP ARG 2000 AEROPARQUE,GU,GRAN BS AS,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,1990.439,2017-01,VERANO
1,2017,1,ABRILHCY,ABRIL CLUB DE CAMPO,GU,GRAN BS AS,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,1609.464,2017-01,VERANO
2,2017,1,ACARQQ3Y,ASOC COOP ARG QUEQUEN,GU,BUENOS AIRES,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,421.334,2017-01,VERANO
3,2017,1,ACARSLSY,ASOC COOP ARG SAN LORENZO,GU,LITORAL,SANTA FE,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,449.145,2017-01,VERANO
4,2017,1,ACERBR1Y,PLANTA BRAGADO,GU,BUENOS AIRES,BUENOS AIRES,GRAN USUARIO MEM,GRAN USUARIO,GUMAS AUTOGENERADORES,INDUSTRIAL COMERCIAL GRANDE,18073.170,2017-01,VERANO


In [51]:


ruta_final = PROCESSED_DATA_DIR / "demanda_limpia.csv"


# Guardamos el csv en la carpeta processed
df_limpio.to_csv(ruta_final)


print(f"Archivo guardado como:")
print(f"Ruta utilizada: {ruta_final}")


Archivo guardado como:
Ruta utilizada: /content/eficiencia_energetica/02_data/processed/demanda_limpia.csv


---


# 4. Exploración de los datos

En esta fase se analizarán los datos ya preparados para encontrar patrones, tendencias o comportamientos relevantes.

El análisis exploratorio permitirá validar o ajustar las hipótesis iniciales y entender mejor la demanda energética antes de construir el modelo.


## 4.1 Consumo energético por año

Se analizará la evolución de la demanda energética a lo largo de los años disponibles en el dataset.


## 4.2 Consumo energético por mes

Se revisará si existen meses con mayor o menor consumo, buscando detectar posibles comportamientos estacionales.


## 4.3 Consumo por categoría

Se comparará el consumo según las categorías disponibles en el dataset, como residencial, comercial o industrial, en caso de que estén presentes.


## 4.4 Validación de hipótesis

Se revisarán las hipótesis planteadas al inicio del trabajo y se evaluará si los datos permiten confirmarlas, rechazarlas o ajustarlas parcialmente.


---


# 5. Modelado

En esta fase se construirá un modelo base para estimar la demanda energética.

La idea no es buscar el modelo más complejo, sino construir una primera solución clara, explicable y alineada con el objetivo del proyecto.


## 5.1 Selección de la variable objetivo

Se definirá cuál será la variable que el modelo intentará estimar.

En este proyecto, la variable objetivo estará relacionada con la demanda o consumo energético.


## 5.2 Selección de variables predictoras

Se elegirán las variables que podrían ayudar al modelo a estimar la demanda.

Estas variables pueden estar relacionadas con el tiempo, la categoría tarifaria u otras columnas disponibles en el dataset.


## 5.3 División de datos de entrenamiento y prueba

Se dividirán los datos en un conjunto de entrenamiento y otro de prueba.

Esto permitirá entrenar el modelo con una parte de los datos y evaluar su comportamiento con datos que no vio durante el entrenamiento.


## 5.4 Entrenamiento del modelo base

Se entrenará un primer modelo simple para generar predicciones iniciales de demanda energética.


---


# 6. Evaluación

En esta fase se analizará qué tan bien funciona el modelo construido.

No alcanza con generar predicciones: también es necesario medir el error, comparar valores reales contra valores estimados e interpretar si el resultado es útil para el objetivo planteado.


## 6.1 Métricas de evaluación

Se calcularán métricas que permitan medir el error del modelo, como MAE, RMSE y R2 si corresponde.


## 6.2 Comparación entre valores reales y predichos

Se compararán los resultados generados por el modelo contra los valores reales del dataset.


## 6.3 Interpretación de resultados

Se explicará en palabras simples si el modelo funciona de manera aceptable, dónde puede fallar y qué limitaciones tiene.


---


# 7. Implantación conceptual

En esta fase se explicará cómo podría utilizarse la solución en un escenario real.

No necesariamente se desarrollará una aplicación completa, pero sí se describirá cómo los resultados del análisis podrían convertirse en una herramienta útil para la toma de decisiones.


## 7.1 Uso esperado de la solución

La solución podría ser utilizada por una distribuidora eléctrica, un equipo técnico o un área de planificación para consultar tendencias de consumo y estimaciones de demanda.


## 7.2 Entrada y salida del sistema

Como entrada, el sistema utilizaría datos históricos de demanda energética.

Como salida, podría mostrar gráficos, indicadores, períodos de mayor consumo y estimaciones generadas por el modelo.


## 7.3 Posibles mejoras futuras

Como mejora futura se podría sumar una interfaz simple, integrar datos climáticos, permitir la carga de nuevos archivos o desplegar una versión web del análisis.


---


# 8. Conclusiones finales

En esta sección se resumirán los principales resultados del trabajo.

Se retomará el problema inicial, se explicará qué se pudo observar en los datos, qué aportó el modelo y qué limitaciones quedaron para futuras mejoras.


## 8.1 Principales hallazgos

Se resumirán los patrones o comportamientos más importantes encontrados durante el análisis.


## 8.2 Limitaciones

Se mencionarán las limitaciones del dataset, del modelo y del alcance general del trabajo.


## 8.3 Próximos pasos

Se dejarán planteadas posibles mejoras para una segunda etapa del proyecto.


---


# 9. Referencias y anexos

En esta sección se incluirán las fuentes utilizadas para el dataset, material de referencia de la metodología CRISP-DM y cualquier anexo que ayude a entender el trabajo.
